## Setup and Data Generation

In [1]:
# Stop any existing session
try:
    spark.stop()
except:
    pass

In [2]:
workshop_name = "file-size-deep-dive"

from pyspark.sql import SparkSession

executor_memory = "8g"
executor_cores = 2
num_executors = 2


spark = SparkSession.builder \
        .appName(workshop_name) \
        .master("spark://spark-master:7077") \
        .config("spark.executor.memory", executor_memory) \
        .config("spark.executor.cores", executor_cores) \
        .config("spark.executor.instances", num_executors) \
        .config("spark.cores.max", executor_cores * num_executors) \
    .getOrCreate()

Setting Spark log level to "WARN".


In [3]:
spark

In [4]:
import os 
import glob
from pathlib import Path

from generate_data import run
from run_ddl import run_ddl 

data_size_gb = 30
overwrite_existing_data = False
data_folder = './data'

# Only when overwrite = False and there are atleast 1 csv files we need to skip data gen (everyother time we need to generate data) 
if not (not overwrite_existing_data and len(glob.glob(os.path.join(data_folder, '*.csv'))) > 1):
    print(f'Creating raw TPCH data in {data_folder}')
    run(scaling_factor=data_size_gb)

print(f'Loading data in raw TPCH folder {data_folder} to Iceberg Tables')
run_ddl(data_path=Path(data_folder), spark=spark, recreate=True) # Load created data into Iceberg tables on Minio (S3 equivalent)

Creating raw TPCH data in ./data


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loading data in raw TPCH folder ./data to Iceberg Tables


In [5]:
%%sql
show tables IN prod.db

26/05/07 19:23:38 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


namespace,tableName,isTemporary
prod.db,customer,False
prod.db,lineitem,False
prod.db,nation,False
prod.db,orders,False
prod.db,part,False
prod.db,partsupp,False
prod.db,region,False
prod.db,supplier,False


In [6]:
%%sql
use prod.db

++
||
++
++

In [7]:
%%sql
DESCRIBE lineitem

col_name,data_type,comment
l_orderkey,int,None
l_partkey,int,None
l_suppkey,int,None
l_linenumber,int,None
l_quantity,double,None
l_extendedprice,double,None
l_discount,double,None
l_tax,double,None
l_returnflag,string,None
l_linestatus,string,None


## Many small files leads to Spark spending a lot of time opening them instead of processing data

In [8]:
! ls -ltha data/lineitem.csv

-rw-r--r-- 1 root root 23G May  7 19:00 data/lineitem.csv


In [10]:
%%sql
SELECT
  MONTH (l_receiptdate) AS receipt_month,
  COUNT(*) AS num_line_items
FROM
  lineitem
GROUP BY
  1
ORDER BY
  2 desc
LIMIT
  10

receipt_month,num_line_items
7,16234226
8,16161841
5,16140024
6,15710469
9,15195007
4,15141057
10,15116206
3,15056278
11,14054000
12,14015957


* In the Spark UI stage tab for the above job we can see that we have 180 tasks
* Spark spends a lot of time opening files as opposed to processing them
* In the UI the green tasks should be long running ideally, not spotty as we see below (see the small input and outputs per tasks).
![Small File Reads](small_files.png)


![Small File IO](small_file_io.png)

#### Let's look at the files that make up the lineitem table

In [21]:
spark.sql("""SELECT 
    ROW_NUMBER() OVER (ORDER BY file_path) AS row_num,
    file_path,
    file_size_in_bytes,
    ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb
FROM prod.db.lineitem.files
ORDER BY 2""").show(200, False)

26/05/07 19:52:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/07 19:52:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/07 19:52:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/07 19:52:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/07 19:52:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+---------------------------------------------------------------------------------------------------+------------------+------------+
|row_num|file_path                                                                                          |file_size_in_bytes|file_size_mb|
+-------+---------------------------------------------------------------------------------------------------+------------------+------------+
|1      |s3://warehouse/prod/db/lineitem/data/00000-382-f552f71c-79bd-4c1b-a63f-f49f4288803e-0-00001.parquet|29391131          |28.03       |
|2      |s3://warehouse/prod/db/lineitem/data/00001-383-f552f71c-79bd-4c1b-a63f-f49f4288803e-0-00001.parquet|29174398          |27.82       |
|3      |s3://warehouse/prod/db/lineitem/data/00002-384-f552f71c-79bd-4c1b-a63f-f49f4288803e-0-00001.parquet|29168935          |27.82       |
|4      |s3://warehouse/prod/db/lineitem/data/00003-385-f552f71c-79bd-4c1b-a63f-f49f4288803e-0-00001.parquet|29158692          |27.81       |
|5    

## Use helper functions to re-size files optimally

In [11]:
%%sql
CREATE TABLE
  prod.db.lineitem_resized AS
SELECT
  *
FROM
  prod.db.lineitem

++
||
++
++

In [12]:
spark.sql("CALL demo.system.rewrite_data_files('prod.db.lineitem_resized')")
# resizes files in the table to 512 MB
# has options to resize per partition, when running this as part of your pipeline
# Delta has OPTIMIZE, which is similar to this Iceberg function

DataFrame[rewritten_data_files_count: int, added_data_files_count: int, rewritten_bytes_count: bigint, failed_data_files_count: int, removed_delete_files_count: int]

In [24]:
%%sql
SELECT
  ROW_NUMBER() OVER (
    ORDER BY
      file_path
  ) AS row_num,
  file_path,
  file_size_in_bytes,
  ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb
FROM
  prod.db.lineitem_resized.files
ORDER BY
  2

26/05/07 19:55:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/07 19:55:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/07 19:55:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/07 19:55:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/07 19:55:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


row_num,file_path,file_size_in_bytes,file_size_mb
1,s3://warehouse/prod/db/lineitem_resized/data/00000-1175-5eda9c55-a655-47e0-a07b-30b1df36ac84-0-00001.parquet,535236450,510.44
2,s3://warehouse/prod/db/lineitem_resized/data/00001-1176-5eda9c55-a655-47e0-a07b-30b1df36ac84-0-00001.parquet,532414293,507.75
3,s3://warehouse/prod/db/lineitem_resized/data/00002-1177-5eda9c55-a655-47e0-a07b-30b1df36ac84-0-00001.parquet,532300889,507.64
4,s3://warehouse/prod/db/lineitem_resized/data/00003-1178-5eda9c55-a655-47e0-a07b-30b1df36ac84-0-00001.parquet,532403825,507.74
5,s3://warehouse/prod/db/lineitem_resized/data/00004-1179-5eda9c55-a655-47e0-a07b-30b1df36ac84-0-00001.parquet,532381409,507.72
6,s3://warehouse/prod/db/lineitem_resized/data/00005-1180-5eda9c55-a655-47e0-a07b-30b1df36ac84-0-00001.parquet,529356133,504.83
7,s3://warehouse/prod/db/lineitem_resized/data/00006-1181-5eda9c55-a655-47e0-a07b-30b1df36ac84-0-00001.parquet,528553209,504.07
8,s3://warehouse/prod/db/lineitem_resized/data/00007-1182-5eda9c55-a655-47e0-a07b-30b1df36ac84-0-00001.parquet,528707202,504.21
9,s3://warehouse/prod/db/lineitem_resized/data/00008-1183-5eda9c55-a655-47e0-a07b-30b1df36ac84-0-00001.parquet,528586748,504.1
10,s3://warehouse/prod/db/lineitem_resized/data/00009-1184-5eda9c55-a655-47e0-a07b-30b1df36ac84-0-00001.parquet,223957502,213.58


In [25]:
%%sql
SELECT
  MONTH (l_receiptdate) AS receipt_month,
  COUNT(*) AS num_line_items
FROM
  lineitem_resized
GROUP BY
  1
ORDER BY
  2 desc
LIMIT
  10

receipt_month,num_line_items
7,16234226
8,16161841
5,16140024
6,15710469
9,15195007
4,15141057
10,15116206
3,15056278
11,14054000
12,14015957


* We can now see Spark spending most of its time processing data as opposed to reading it
* We can see this as large green blocks in the stage tab
![512MB File Reads](mb512_files.png)
![512MB File IO](mb512_file_io.png)

#### Exercise

Why are the inputs only 7MB, when the files are 512MB? 

## File size can be set as a TBLPROPERTY, but requires code tuning

* Lets create a table and set partition size property, which indicates the size of a file on disk.

In [29]:
%%sql
CREATE TABLE
  IF NOT EXISTS prod.db.lineitem_part_year_2gb_intask_mem (
    l_orderkey BIGINT,
    l_partkey BIGINT,
    l_suppkey BIGINT,
    l_linenumber INT,
    l_quantity DECIMAL(15, 2),
    l_extendedprice DECIMAL(15, 2),
    l_discount DECIMAL(15, 2),
    l_tax DECIMAL(15, 2),
    l_returnflag STRING,
    l_linestatus STRING,
    l_shipdate DATE,
    l_commitdate DATE,
    l_receiptdate DATE,
    l_shipinstruct STRING,
    l_shipmode STRING,
    l_comment STRING
  ) USING iceberg PARTITIONED BY (YEAR (l_shipdate)) TBLPROPERTIES (
    'format-version' = '2',
    'write.spark.advisory-partition-size-bytes' = '2147483648', -- 2 GB
    'write.target-file-size-bytes' = '536870912' -- 512 MB
  );

++
||
++
++

* The propery `write.spark.advisory-partition-size-bytes` indicates the size of data in memory before Spark-Iceberg writer inserts data into this table
* The propery `write.target-file-size-bytes` indicates the max size of data on disk. 
* **Note** When we write data in memory to disk there will be significant size reduction, due to parquet compression. Think > 3X size compression. 
* The table is also partitioned by year(shipdate).
* Let's insert data into this table, and see the file sizes.

In [30]:
%%sql
INSERT INTO
  prod.db.lineitem_part_year_2gb_intask_mem
SELECT
  *
FROM
  prod.db.lineitem

++
||
++
++

In [36]:
%%sql
SELECT
  ROW_NUMBER() OVER (
    ORDER BY
      file_path
  ) AS row_num,
  file_path,
  file_size_in_bytes,
  ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb,
  split(file_path, '-')[1] AS task_that_wrote_this_file
FROM
  prod.db.lineitem_part_year_2gb_intask_mem.files
ORDER BY
  2, 3

26/05/07 20:36:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/07 20:36:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/07 20:36:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/07 20:36:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/07 20:36:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


row_num,file_path,file_size_in_bytes,file_size_mb,task_that_wrote_this_file
1,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1992/00004-1699-3cea6739-b166-42a5-acc4-a3e7227b3a04-0-00001.parquet,535671069,510.86,1699
2,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1992/00004-1699-3cea6739-b166-42a5-acc4-a3e7227b3a04-0-00002.parquet,40140187,38.28,1699
3,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1993/00005-1700-3cea6739-b166-42a5-acc4-a3e7227b3a04-0-00001.parquet,535437633,510.63,1700
4,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1993/00005-1700-3cea6739-b166-42a5-acc4-a3e7227b3a04-0-00002.parquet,37774919,36.02,1700
5,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1993/00006-1701-3cea6739-b166-42a5-acc4-a3e7227b3a04-0-00001.parquet,119297202,113.77,1701
6,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1994/00009-1704-3cea6739-b166-42a5-acc4-a3e7227b3a04-0-00001.parquet,535393884,510.59,1704
7,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1994/00009-1704-3cea6739-b166-42a5-acc4-a3e7227b3a04-0-00002.parquet,37558536,35.82,1704
8,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1994/00010-1705-3cea6739-b166-42a5-acc4-a3e7227b3a04-0-00001.parquet,119319915,113.79,1705
9,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1995/00007-1702-3cea6739-b166-42a5-acc4-a3e7227b3a04-0-00001.parquet,535703913,510.89,1702
10,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1995/00007-1702-3cea6739-b166-42a5-acc4-a3e7227b3a04-0-00002.parquet,41489553,39.57,1702


In [40]:
%%sql
SELECT
  split(file_path, '/')[7] AS partition,
  ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb,
  split(file_path, '-')[1] AS task_that_wrote_this_file
FROM
  prod.db.lineitem_part_year_2gb_intask_mem.files
ORDER BY
  1

partition,file_size_mb,task_that_wrote_this_file
l_shipdate_year=1992,510.86,1699
l_shipdate_year=1992,38.28,1699
l_shipdate_year=1993,510.63,1700
l_shipdate_year=1993,36.02,1700
l_shipdate_year=1993,113.77,1701
l_shipdate_year=1994,510.59,1704
l_shipdate_year=1994,35.82,1704
l_shipdate_year=1994,113.79,1705
l_shipdate_year=1995,510.89,1702
l_shipdate_year=1995,39.57,1702


![Iceberg Task Write wo Spark Partition](task_file_map_no_part.png)
![Iceberg Task Write wo Spark Partition](task_file_map_no_part_ui.png)

* What's happening here is that the data from `lineitem` table is not partitioned when it arrives in the iceberg write task.
* An iceberg write task got input for year 1997 & 1996 so it only had enough data for ~200MB and ~180MB
![Iceberg Task Write](iceberg_task_write.png)
* If we partition the data in Spark before writing to disk, we can avoid this issue.

In [32]:
%%sql
CREATE TABLE
  IF NOT EXISTS prod.db.lineitem_part_year_2gb_intask_mem_code_part (
    l_orderkey BIGINT,
    l_partkey BIGINT,
    l_suppkey BIGINT,
    l_linenumber INT,
    l_quantity DECIMAL(15, 2),
    l_extendedprice DECIMAL(15, 2),
    l_discount DECIMAL(15, 2),
    l_tax DECIMAL(15, 2),
    l_returnflag STRING,
    l_linestatus STRING,
    l_shipdate DATE,
    l_commitdate DATE,
    l_receiptdate DATE,
    l_shipinstruct STRING,
    l_shipmode STRING,
    l_comment STRING
  ) USING iceberg PARTITIONED BY (YEAR (l_shipdate)) TBLPROPERTIES (
    'format-version' = '2',
    'write.spark.advisory-partition-size-bytes' = '2147483648', -- 2 GB
    'write.target-file-size-bytes' = '536870912' -- 512 MB
  );

++
||
++
++

In [33]:
# This forces same-year rows to same task
import pyspark.sql.functions as F

spark.table("prod.db.lineitem")\
  .repartition(F.year(F.col("l_shipdate"))) \
  .writeTo("prod.db.lineitem_part_year_2gb_intask_mem_code_part") \
  .append()

In [35]:
%%sql
SELECT
  ROW_NUMBER() OVER (
    ORDER BY
      file_path
  ) AS row_num,
  file_path,
  file_size_in_bytes,
  ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb,
  split(file_path, '-')[1] AS task_that_wrote_this_file
FROM
  prod.db.lineitem_part_year_2gb_intask_mem_code_part.files
ORDER BY
  2, 3

26/05/07 20:36:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/07 20:36:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/07 20:36:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/07 20:36:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/07 20:36:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


row_num,file_path,file_size_in_bytes,file_size_mb,task_that_wrote_this_file
1,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem_code_part/data/l_shipdate_year=1992/00003-1899-af95f8f9-74d4-4278-9398-c6a2b5cd7e57-0-00001.parquet,535620669,510.81,1899
2,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem_code_part/data/l_shipdate_year=1992/00003-1899-af95f8f9-74d4-4278-9398-c6a2b5cd7e57-0-00002.parquet,40110632,38.25,1899
3,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem_code_part/data/l_shipdate_year=1993/00004-1898-af95f8f9-74d4-4278-9398-c6a2b5cd7e57-0-00001.parquet,535438925,510.63,1898
4,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem_code_part/data/l_shipdate_year=1993/00004-1898-af95f8f9-74d4-4278-9398-c6a2b5cd7e57-0-00002.parquet,157156466,149.88,1898
5,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem_code_part/data/l_shipdate_year=1994/00006-1901-af95f8f9-74d4-4278-9398-c6a2b5cd7e57-0-00001.parquet,535424403,510.62,1901
6,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem_code_part/data/l_shipdate_year=1994/00006-1901-af95f8f9-74d4-4278-9398-c6a2b5cd7e57-0-00002.parquet,156954383,149.68,1901
7,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem_code_part/data/l_shipdate_year=1995/00005-1900-af95f8f9-74d4-4278-9398-c6a2b5cd7e57-0-00001.parquet,535694637,510.88,1900
8,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem_code_part/data/l_shipdate_year=1995/00005-1900-af95f8f9-74d4-4278-9398-c6a2b5cd7e57-0-00002.parquet,161948144,154.45,1900
9,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem_code_part/data/l_shipdate_year=1996/00002-1897-af95f8f9-74d4-4278-9398-c6a2b5cd7e57-0-00001.parquet,535498545,510.69,1897
10,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem_code_part/data/l_shipdate_year=1996/00002-1897-af95f8f9-74d4-4278-9398-c6a2b5cd7e57-0-00002.parquet,154900358,147.72,1897


In [42]:
%%sql
SELECT
  split(file_path, '/')[7] AS partition,
  ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb,
  split(file_path, '-')[1] AS task_that_wrote_this_file
FROM
  prod.db.lineitem_part_year_2gb_intask_mem_code_part.files
ORDER BY
  3

partition,file_size_mb,task_that_wrote_this_file
l_shipdate_year=1998,492.13,1895
l_shipdate_year=1997,510.7,1896
l_shipdate_year=1997,146.56,1896
l_shipdate_year=1996,510.69,1897
l_shipdate_year=1996,147.72,1897
l_shipdate_year=1993,510.63,1898
l_shipdate_year=1993,149.88,1898
l_shipdate_year=1992,510.81,1899
l_shipdate_year=1992,38.25,1899
l_shipdate_year=1995,510.88,1900


![Iceberg Task Write w Spark Partition](task_file_map_part.png)
![Iceberg Task Write w Spark Partition](task_file_map_part_ui.png)

## Vendors do this (& more) for free

* Data maintenance is a place where vendors do a great job.
* Both Snowflake and Databricks have automated file resizing (along with cleaning up catalogs, deleting old files, etc).
* If you don't want to do maintenance (or want much simpler management in general) vendors are a good choice.

| Concern | Databricks Unity Catalog | Snowflake |
|---|---|---|
| File compaction | ✅ Predictive Optimization | ✅ automatic |
| Old snapshot cleanup | ✅ automatic | ✅ automatic |
| Orphan file removal | ✅ automatic | ✅ automatic |
| Manifest compaction | N/A | ✅ automatic |

* Both dbx and Snowflake have sane defaults for data retention (which dictates file removal) which you can change as needed.

## Ignore - ScratchPad

Image to compare multiple small tasks vs a few large task in Spark UI and their time taken
output num rows

* parquet row group 128MB
* small files -> storage problem and job reading from table 
* file size target 512 MB  (4 row groups per file) 
* avoid dynamic partition, produce tasks * partitions = file_count
* repart or sort before write 
* use parquet 
* with SPark 3.3 and iceberg v1.1 > sort and distribution are handled for you 
* default write.target-file-size-bytes 512MB
* 1 task -> 1 file 

In [ ]:
%%sql
CREATE TABLE
  IF NOT EXISTS prod.db.lineitem_part_year (
    l_orderkey BIGINT,
    l_partkey BIGINT,
    l_suppkey BIGINT,
    l_linenumber INT,
    l_quantity DECIMAL(15, 2),
    l_extendedprice DECIMAL(15, 2),
    l_discount DECIMAL(15, 2),
    l_tax DECIMAL(15, 2),
    l_returnflag STRING,
    l_linestatus STRING,
    l_shipdate DATE,
    l_commitdate DATE,
    l_receiptdate DATE,
    l_shipinstruct STRING,
    l_shipmode STRING,
    l_comment STRING
  ) USING iceberg PARTITIONED BY (YEAR (l_shipdate)) TBLPROPERTIES ('format-version' = '2')

In [ ]:
%%sql
INSERT INTO
  prod.db.lineitem_part_year
SELECT
  *
FROM
  prod.db.lineitem

In [ ]:
%%sql
SELECT
  ROW_NUMBER() OVER (
    ORDER BY
      file_path
  ) AS row_num,
  file_path,
  file_size_in_bytes,
  ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb
FROM
  prod.db.lineitem_part_year.files
ORDER BY
  1

In [ ]:
print(spark.conf.get("spark.sql.adaptive.advisoryPartitionSizeInBytes"))

In [ ]:
%%sql
CREATE TABLE
  IF NOT EXISTS prod.db.lineitem_part_year_4 (
    l_orderkey BIGINT,
    l_partkey BIGINT,
    l_suppkey BIGINT,
    l_linenumber INT,
    l_quantity DECIMAL(15, 2),
    l_extendedprice DECIMAL(15, 2),
    l_discount DECIMAL(15, 2),
    l_tax DECIMAL(15, 2),
    l_returnflag STRING,
    l_linestatus STRING,
    l_shipdate DATE,
    l_commitdate DATE,
    l_receiptdate DATE,
    l_shipinstruct STRING,
    l_shipmode STRING,
    l_comment STRING
  ) USING iceberg PARTITIONED BY (YEAR (l_shipdate)) TBLPROPERTIES (
    'format-version' = '2',
    'write.spark.advisory-partition-size-bytes' = '1073741824',
    'write.target-file-size-bytes' = '536870912'
  )
  -- 1 GB

In [ ]:
%%sql
INSERT INTO
  prod.db.lineitem_part_year_4
SELECT
  *
FROM
  prod.db.lineitem

In [ ]:
%%sql
SELECT
  ROW_NUMBER() OVER (
    ORDER BY
      file_path
  ) AS row_num,
  file_path,
  file_size_in_bytes,
  ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb
FROM
  prod.db.lineitem_part_year_4.files
ORDER BY
  1

* parquet compression is very good see task in -> out

In [ ]:
%%sql
CREATE TABLE
  IF NOT EXISTS prod.db.lineitem_part_year_5 (
    l_orderkey BIGINT,
    l_partkey BIGINT,
    l_suppkey BIGINT,
    l_linenumber INT,
    l_quantity DECIMAL(15, 2),
    l_extendedprice DECIMAL(15, 2),
    l_discount DECIMAL(15, 2),
    l_tax DECIMAL(15, 2),
    l_returnflag STRING,
    l_linestatus STRING,
    l_shipdate DATE,
    l_commitdate DATE,
    l_receiptdate DATE,
    l_shipinstruct STRING,
    l_shipmode STRING,
    l_comment STRING
  ) USING iceberg PARTITIONED BY (YEAR (l_shipdate)) TBLPROPERTIES (
    'format-version' = '2',
    'write.spark.advisory-partition-size-bytes' = '2147483648',
    'write.target-file-size-bytes' = '536870912'
  )
  -- 2 GB task file size

In [ ]:
%%sql
INSERT INTO
  prod.db.lineitem_part_year_5
SELECT
  *
FROM
  prod.db.lineitem

In [ ]:
%%sql
SELECT
  ROW_NUMBER() OVER (
    ORDER BY
      file_path
  ) AS row_num,
  file_path,
  file_size_in_bytes,
  ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb
FROM
  prod.db.lineitem_part_year_5.files
ORDER BY
  2

In [ ]:
%%sql
CREATE TABLE
  IF NOT EXISTS prod.db.lineitem_part_year_6 (
    l_orderkey BIGINT,
    l_partkey BIGINT,
    l_suppkey BIGINT,
    l_linenumber INT,
    l_quantity DECIMAL(15, 2),
    l_extendedprice DECIMAL(15, 2),
    l_discount DECIMAL(15, 2),
    l_tax DECIMAL(15, 2),
    l_returnflag STRING,
    l_linestatus STRING,
    l_shipdate DATE,
    l_commitdate DATE,
    l_receiptdate DATE,
    l_shipinstruct STRING,
    l_shipmode STRING,
    l_comment STRING
  ) USING iceberg PARTITIONED BY (YEAR (l_shipdate)) TBLPROPERTIES (
    'format-version' = '2',
    'write.spark.advisory-partition-size-bytes' = '2147483648',
    'write.target-file-size-bytes' = '536870912'
  )
  -- 2 GB task file size

In [ ]:
# This forces same-year rows to same task
import pyspark.sql.functions as F

spark.table("prod.db.lineitem")\
  .repartition(F.year(F.col("l_shipdate"))) \
  .writeTo("prod.db.lineitem_part_year_6") \
  .append()

In [ ]:
%%sql
SELECT
  ROW_NUMBER() OVER (
    ORDER BY
      file_path
  ) AS row_num,
  file_path,
  file_size_in_bytes,
  ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb
FROM
  prod.db.lineitem_part_year_6.files
ORDER BY
  2

* delta 

In [ ]:
spark

ui plan for iceberg write shuffle hash part

In [ ]:
%%sql
SELECT
  MONTH (l_receiptdate) AS receipt_month,
  COUNT(*) AS num_line_items
FROM
  lineitem_part_year
GROUP BY
  1
ORDER BY
  2 desc
LIMIT
  10

* delta -- SKIP

In [5]:
%%sql
show catalogs

catalog
delta_catalog
spark_catalog


In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS delta_catalog.db LOCATION 's3://warehouse/delta/db/'

In [ ]:
spark.sql("CREATE namespace IF NOT EXISTS delta_catalog.db")

In [ ]:
%%sql
CREATE TABLE
  IF NOT EXISTS delta_catalog.db.lineitem_part_year (
    l_orderkey BIGINT,
    l_partkey BIGINT,
    l_suppkey BIGINT,
    l_linenumber INT,
    l_quantity DECIMAL(15, 2),
    l_extendedprice DECIMAL(15, 2),
    l_discount DECIMAL(15, 2),
    l_tax DECIMAL(15, 2),
    l_returnflag STRING,
    l_linestatus STRING,
    l_shipdate DATE,
    l_commitdate DATE,
    l_receiptdate DATE,
    l_shipinstruct STRING,
    l_shipmode STRING,
    l_comment STRING
  ) USING delta PARTITIONED BY (YEAR (l_shipdate)) LOCATION 's3://warehouse/delta/db/lineitem_part_year';

In [ ]:
%%sql
CREATE TABLE
  IF NOT EXISTS delta_catalog.people10m (
    id INT,
    firstName STRING,
    middleName STRING,
    lastName STRING,
    gender STRING,
    birthDate TIMESTAMP,
    ssn STRING,
    salary INT
  ) USING DELTA